In [1]:
from pydantic import BaseModel, Field
from typing import List
from openai import OpenAI
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os
import time
from langchain_core.prompts import ChatPromptTemplate
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=os.getenv("MODEL_BASE_URL")
)
DATA_FILE_PATH=os.getenv("DATA_FILE_PATH")
DATA_TEST_FILE_PATH=os.getenv("DATA_TEST_FILE_PATH")
DATA_EVAL_ANSWER_FILE_PATH=os.getenv("DATA_EVAL_ANSWER_FILE_PATH")


class VehicleIssue(BaseModel):
    issue_id: str = Field(description="Unique identifier, e.g. 'p0420-001'")
    issue_name: str = Field(description="Name of the issue, e.g. 'Catalyst System Efficiency Below Threshold'")
    obd_code: str = Field(description="Real OBD-II code, e.g. 'P0420', 'P0300'. Use 'N/A' if not code-based")
    system: str = Field(description="Engine, Transmission, Brakes, Electrical, Suspension, Cooling, etc.")
    component: str = Field(description="Specific part, e.g. 'Catalytic Converter', 'Brake Pads', 'Alternator'")
    severity: str = Field(description="Low, Medium, High, Critical")
    symptoms: str = Field(description="Comma-separated list, e.g. 'Check engine light, rough idle, reduced power'")
    likely_causes: str = Field(description="Comma-separated list, e.g. 'Faulty O2 sensor, worn spark plugs'")
    diagnostic_steps: str = Field(description="Step-by-step instructions to confirm the diagnosis")
    diy_or_mechanic: str = Field(description="DIY, Mechanic Recommended, Mechanic Required")

class VehicleIssueDataset(BaseModel):
    issues: List[VehicleIssue]

In [3]:
all_issues = []
batch_size = 10
num_batches = 5

def generate_batch(batch_prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.parse(
                model=os.getenv("AI_MODEL"),
                messages=[{"role": "user", "content": batch_prompt}],
                response_format=VehicleIssueDataset,
                reasoning_effort="low",
            )
            return response.choices[0].message.parsed.issues
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(1)
    print("  All retries failed, skipping batch.")
    return []

In [4]:


for i in range(num_batches):
    batch_prompt = f"""
Generate exactly {batch_size} diverse, realistic vehicle issues (batch {i+1} of {num_batches}).
Cover different systems (engine, transmission, brakes, electrical, suspension, cooling).
Use real OBD-II codes where applicable (e.g. P0171, P0300, P0420, P0455), 'N/A' for non-code issues.
Vary severity levels and mix DIY-fixable issues with ones that require a mechanic.
Output valid JSON only — field names must exactly match the schema, with no markdown formatting (no asterisks, no bold) in keys or values.
Avoid generating an issue with the same issue_name and obd_code as any previously generated issue.
Previously generated issues:
{[(iss.issue_name, iss.obd_code) for iss in all_issues]}
""".strip()

    batch_issues = generate_batch(batch_prompt)
    print(f"Batch {i+1}: requested {batch_size}, got {len(batch_issues)}")
    all_issues.extend(batch_issues)

df = pd.DataFrame([issue.model_dump() for issue in all_issues])

# Remove duplicate issues
df = df.drop_duplicates(
    subset=["issue_name", "obd_code"],
    keep="first"
)

# Give unique IDs AFTER removing duplicates
df["issue_id"] = [
    f"issue_{i:04d}"
    for i in range(1, len(df) + 1)
]

df.to_csv(DATA_FILE_PATH, index=False)

print(f"Generated {len(df)} unique vehicle issues")
print(f"Unique IDs: {df['issue_id'].nunique()}")



Batch 1: requested 10, got 10
Batch 2: requested 10, got 10
Batch 3: requested 10, got 10
Batch 4: requested 10, got 10
Batch 5: requested 10, got 10
Generated 50 unique vehicle issues
Unique IDs: 50


In [5]:
class GroundTruthRetrival(BaseModel):
    """A test question with expected keywords and reference answer."""

    question: str = Field(description="The question to ask the RAG system")
    keywords: list[str] = Field(description="Keywords that must appear in retrieved context")
    reference_answer: str = Field(description="The reference answer for this question")
    category: str = Field(description="Question category (e.g., direct_fact, spanning, temporal)")

In [6]:

import time

def parse_with_retry(prompt, model, max_retries=2, delay=2):
    for attempt in range(max_retries + 1):
        try:
            response = openai_client.chat.completions.parse(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format=GroundTruthBatch,
            )
            return response.choices[0].message.parsed.questions
        except Exception as e:
            if attempt == max_retries:
                raise
            print(f"Attempt {attempt + 1} failed: {e}. Retrying...")
            time.sleep(delay)
class GroundTruthBatch(BaseModel):
    questions: List[GroundTruthRetrival]


prompt1_template = """
You are a vehicle diagnostic expert generating evaluation questions for a RAG system.

Read the vehicle issue described below and generate exactly 2 test questions
that a real user might ask about this issue.

For each question, also provide:
- keywords: 3-6 important terms that MUST appear in the retrieved context to answer this question correctly
- reference_answer: a concise, accurate answer to the question, grounded only in the issue details below
- category: one of "direct_fact", "spanning", "temporal", "diagnostic", "causal"

Issue details:
{content}
""".strip()

from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

import os
import json
import glob
from tqdm.auto import tqdm

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=os.getenv("MODEL_BASE_URL")
)


results = []

df = pd.read_csv(DATA_FILE_PATH)
results = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    d = row.to_dict()
    d["content"] = "\n".join(f"{k}: {v}" for k, v in row.items())
    prompt = prompt1_template.format(**d)
    try:
        questions = parse_with_retry(prompt, os.getenv("AI_MODEL"))
    except Exception as e:
        print(f"Failed for generate data {e}")
        continue
    for q in questions:
        record = q.model_dump()
        results.append(record)
with open(DATA_TEST_FILE_PATH, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"Wrote {len(results)} questions to {DATA_TEST_FILE_PATH}")

  0%|          | 0/50 [00:00<?, ?it/s]

Wrote 100 questions to data/ground-truth-retrieval.jsonl


In [2]:
import json
import pandas as pd
import sys
from pathlib import Path
from tqdm.auto import tqdm
# Add the v2 directory to path so we can import the rag module
import sys
import os
# 1. Change the notebook's working directory to the repository root 
# (We only do this if we are currently inside the 'project' folder)
if os.path.basename(os.getcwd()) == "project":
    os.chdir("..")
# 2. Add the correct path so we can import rag
sys.path.append(os.path.abspath('project/versions/v2'))
from langchain_openai import ChatOpenAI
from rag import query

REPO_ROOT = Path.cwd()  # since your guard already ensures this is the repo root
DATA_TEST_FILE_PATH = REPO_ROOT / "project" / DATA_TEST_FILE_PATH
# Load the ground truth JSONL file into df_question
df_question = pd.read_json(DATA_TEST_FILE_PATH, lines=True)

# Sample 3 questions to evaluate
df_sample = df_question.sample(n=3, random_state=1)
sample = df_sample.to_dict(orient="records")
evaluations = []
for record in tqdm(sample):
    question = record["question"]
    answer_llm = query(question)
    evaluations.append((record, answer_llm["answer"], answer_llm["relevance"],answer_llm["relevance_explanation"]))


Found existing index at D:\vehicle_assistant_v2\project\dbs\issues.db, loading it.
Loaded 50 documents
Found existing vectorstore at D:\vehicle_assistant_v2\project\dbs\chroma_db, loading it.


  0%|          | 0/3 [00:00<?, ?it/s]


[EXPAND] Expanding query: 'What are the common causes of the P0456 EVAP system leak detected (small leak) code and how can I repair them?'
['P0456 OBD-II EVAP system small leak diagnosis causes repair']

[ROUTER] Evaluating: score=0.53, retries=0/2, docs=5
[ROUTER] -> GENERATE (relevant docs found)

[GENERATE] Creating answer from 5 documents...
[GENERATE] Answer generated

[EXPAND] Expanding query: 'What does the OBD code P0608 indicate?'
['P0608 OBD-II diagnostic trouble code meaning and ECU internal memory error explanation']

[ROUTER] Evaluating: score=0.25, retries=0/2, docs=1
[ROUTER] -> GENERATE (relevant docs found)

[GENERATE] Creating answer from 1 documents...
[GENERATE] Answer generated

[EXPAND] Expanding query: 'How can I diagnose a Camshaft Position Sensor Bank 1 failure?'
['Camshaft Position Sensor Bank 1 diagnostic procedures OBD-II P0340 troubleshooting']

[ROUTER] Evaluating: score=0.53, retries=0/2, docs=5
[ROUTER] -> GENERATE (relevant docs found)

[GENERATE] Crea

In [3]:
# 1. Create the DataFrame from your evaluations
df_eval = pd.DataFrame(evaluations, columns=["record", "answer", "relevance", "explanation"])
# 2. Extract the question from the 'record' column
df_eval["question"] = df_eval.record.apply(lambda d: d["question"])
# 3. Filter and reorder to ONLY keep the 4 columns you want
df_eval = df_eval[["question", "answer", "relevance", "explanation"]]
# 4. Display the relevance stats
df_eval.relevance.value_counts(normalize=True)



relevance
RELEVANT    1.0
Name: proportion, dtype: float64

In [4]:

import os
model_name = os.getenv("AI_MODEL")
REPO_ROOT = Path.cwd()  # since your guard already ensures this is the repo root
DATA_EVAL_ANSWER_FILE_PATH = REPO_ROOT / "project" / DATA_EVAL_ANSWER_FILE_PATH
out_path = f"{DATA_EVAL_ANSWER_FILE_PATH}/{model_name}.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_eval.to_csv(out_path, index=False)

